# City GPT
Character-level Transformer trained on GeoNames allCountries data.

**Runtime → Change runtime type → T4 GPU** before running.

Data lives in ephemeral Colab storage (rebuilt each session).  
Model checkpoints are saved to Google Drive and reloaded automatically.

In [ ]:
# ── 1. Install uv and clone the repo ───────────────────────────────────────
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ["PATH"] += ":/root/.local/bin"

!git clone https://github.com/igui/city-gpt.git
%cd city-gpt
!uv sync --quiet

In [ ]:
# ── 2. Mount Google Drive (checkpoints only) ───────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_CKPT = "/content/drive/MyDrive/city-gpt/city_gpt.pt"
os.makedirs(os.path.dirname(DRIVE_CKPT), exist_ok=True)
print(f"Checkpoints will be saved to: {DRIVE_CKPT}")

In [ ]:
# ── 3. Patch MODEL_PATH to point at Drive ─────────────────────────────────
# We monkey-patch the constant before importing anything else so every
# save/load in city_gpt.py transparently uses Drive.
import importlib, sys

# Remove cached module if re-running this cell
if 'city_gpt' in sys.modules:
    del sys.modules['city_gpt']

import city_gpt
city_gpt.MODEL_PATH = DRIVE_CKPT
print(f"MODEL_PATH = {city_gpt.MODEL_PATH}")
print(f"device     = {city_gpt.device}")

In [ ]:
# ── 4. Download + prepare data (ephemeral, ~5-10 min) ─────────────────────
# Re-run this cell at the start of every new Colab session.
# Pass --max-rows to use a subset, e.g. 1_000_000 for a quick test.
!uv run city_gpt.py prepare
# !uv run city_gpt.py prepare --max-rows 1000000

In [ ]:
# ── 5. Train ───────────────────────────────────────────────────────────────
# If a checkpoint already exists on Drive it will be loaded automatically
# by the generate command; training always starts fresh from random weights.
# To resume, load the checkpoint manually before calling cmd_train:
#
#   import torch
#   ckpt = torch.load(DRIVE_CKPT, map_location=city_gpt.device)
#   model = city_gpt.GPTLanguageModel(ckpt['stoi'].__len__()).to(city_gpt.device)
#   model.load_state_dict(ckpt['model_state'])
#
import argparse
args = argparse.Namespace()   # cmd_train takes an argparse.Namespace (unused fields)
city_gpt.cmd_train(args)

In [ ]:
# ── 6. Generate samples ────────────────────────────────────────────────────
args = argparse.Namespace(prompt="", n=500, temperature=1.0, top_k=None)
city_gpt.cmd_generate(args)

In [ ]:
# ── 7. Generate with a prompt (steer by country + feature type) ────────────
# Prompt format:  <flag><class-emoji><feature-code>|
# Examples:
#   🇺🇸🏙️PPL|   -> US populated place
#   🇯🇵⛰️MT|    -> Japanese mountain
#   🇩🇪💧LK|    -> German lake
args = argparse.Namespace(
    prompt="🇺🇸🏙️PPL|",
    n=200,
    temperature=0.8,
    top_k=50,
)
city_gpt.cmd_generate(args)